In [1]:
import sys
import os
import slurm_utils

HOME = os.environ['HOME']
BASECODE_PATH = os.path.join(HOME,'international-brain-lab/prior-localization/decoding')
if not BASECODE_PATH in sys.path:                                                                              
     sys.path.insert(0, BASECODE_PATH)

fb = open('slurm_decode_base.py','r')
filestr = fb.read()
fb.close()
exec(filestr)

fiteidstr = """
sessdf = dut.query_sessions(selection=SESS_CRITERION)
sessdf = sessdf.sort_values('subject').set_index(['subject', 'eid'])
eid = sys.argv[1]
ADD_TO_SAVING_PATH = sys.argv[2]
print(eid)
if NULL_TYPE == 'impostor-session':
    IMPOSTOR_PATH = os.path.join(OUTPUT_PATH, 
                             dut.decoding_details(TARGET,MODEL,SCORE,
                             ESTIMATORSTR,
                             ALIGN_TIME,
                             CONTROL_FEATURES,
                             N_PSEUDO,NULL_TYPE,TIME_WINDOW,
                             ADD_TO_SAVING_PATH,
                             USE_FAKE_DATA=USE_FAKE_DATA), 
                             '_'.join(['impostors',''])+'.pkl')
    with open(IMPOSTOR_PATH,'rb') as fimpostor:
        impostordict = pickle.load(fimpostor)
else:
    impostordict = None
filenames = fit_eid(eid,sessdf,impostordict = impostordict)
print('saving to files:',filenames)
"""

ADD_TO_SAVING_PATH = 'testnulls1'
SUBDIRECTORY = dut.decoding_details(TARGET,MODEL,SCORE,
                             ESTIMATORSTR,
                             ALIGN_TIME,
                             CONTROL_FEATURES,
                             N_PSEUDO,NULL_TYPE,TIME_WINDOW,
                             ADD_TO_SAVING_PATH,
                             USE_FAKE_DATA=USE_FAKE_DATA)
SLURM_DIRECTORY = os.path.join(GROUP_HOME,'bensonb','international-brain-lab/prior-localization/slurm/')
OUTPUT_PATH = os.path.join(GROUP_HOME,'bensonb/international-brain-lab/prior-localization/decoding/')
print(SUBDIRECTORY)

IMPOSTOR_PATH = os.path.join(OUTPUT_PATH, 
                             SUBDIRECTORY, 
                             '_'.join(['impostors',''])+'.pkl')
if os.path.exists(IMPOSTOR_PATH):
    print('Impostor sessions already exist.')

decode_choice_task_Logistic_accuracy_control_100_impostor-session_align_firstMovement_times_timeWin_-0_1_0_0_testnulls1


In [2]:
# make python file
py_file = os.path.join(OUTPUT_PATH, SUBDIRECTORY, '_'.join(['slurm_decode_eid',''])+'.py')
if not os.path.exists(os.path.join(OUTPUT_PATH, SUBDIRECTORY)):
    os.mkdir(os.path.join(OUTPUT_PATH, SUBDIRECTORY))
with open(py_file,'w') as fp:
    fp.write(filestr)
    fp.write(fiteidstr)
    
sessdf = dut.query_sessions(selection=SESS_CRITERION)
sessdf = sessdf.sort_values('subject').set_index(['subject', 'eid'])
sessdf_eids = sessdf.index.unique(level='eid')

# sessdf_eids=np.unique(sessdf_eids)[30:32]
# sessdf_eids= ['dfd8e7df-dc51-4589-b6ca-7baccfeb94b4', '034e726f-b35f-41e0-8d6c-a22cc32391fb',
#             '7939711b-8b4d-4251-b698-b97c1eaa846e', 'fa704052-147e-46f6-b190-a65b837e605e',
#             '46794e05-3f6a-4d35-afb3-9165091a5a74', '1538493d-226a-46f7-b428-59ce5f43f0f9',
#             'b52182e7-39f6-4914-9717-136db589706e', '89f0d6ff-69f4-45bc-b89e-72868abb042a',
#             '3ce452b3-57b4-40c9-885d-1b814036e936', '2d5f6d81-38c4-4bdc-ac3c-302ea4d5f46e',
#             '4b7fbad4-f6de-43b4-9b15-c7c7ef44db4b', 'd839491f-55d8-4cbe-a298-7839208ba12b',
#             'd2918f52-8280-43c0-924b-029b2317e62c', 'c99d53e6-c317-4c53-99ba-070b26673ac4',
#             'ecb5520d-1358-434c-95ec-93687ecd1396', '5386aba9-9b97-4557-abcd-abc2da66b863',
#             '4b00df29-3769-43be-bb40-128b1cba6d35', '3663d82b-f197-4e8b-b299-7b803a155b84',
#             '85dc2ebd-8aaf-46b0-9284-a197aee8b16f', '15f742e1-1043-45c9-9504-f1e8a53c1744']

# collect all target vectors for impostor sessions
if not os.path.exists(IMPOSTOR_PATH):
    fimpostor = open(IMPOSTOR_PATH, 'wb')
    impostordict = {}
    n_eid = 0
    for eid in sessdf_eids:
        print(n_eid, len(sessdf_eids), eid)
        n_eid += 1
        impostordict[eid] = get_target_vector_eid(eid, sessdf)
    pickle.dump(impostordict, fimpostor)
    fimpostor.close()
else:
    with open(IMPOSTOR_PATH,'rb') as fimpostor:
        impostordict = pickle.load(fimpostor)

fit_metadata['submitted_eids'] = sessdf_eids
fit_metadata['impostor_dictionary'] = impostordict
fn = os.path.join(OUTPUT_PATH,SUBDIRECTORY,'DATE_results')
fn = fn + '.parquet'
metadata_df = pd.Series({'filename': fn, **fit_metadata})
metadata_fn = '.'.join([fn.split('.')[0], 'metadata', 'pkl'])
metadata_fn = metadata_fn.replace('DATE',str(date.today()))
metadata_df.to_pickle(metadata_fn)
print('created metadata file and impostor sessions')

0 334 034e726f-b35f-41e0-8d6c-a22cc32391fb
1 334 7939711b-8b4d-4251-b698-b97c1eaa846e
2 334 46794e05-3f6a-4d35-afb3-9165091a5a74
3 334 dfd8e7df-dc51-4589-b6ca-7baccfeb94b4
4 334 fa704052-147e-46f6-b190-a65b837e605e
5 334 b52182e7-39f6-4914-9717-136db589706e
6 334 3ce452b3-57b4-40c9-885d-1b814036e936
7 334 1538493d-226a-46f7-b428-59ce5f43f0f9
8 334 89f0d6ff-69f4-45bc-b89e-72868abb042a


Inconsistent dimensions for object: None 
(948,),    rewardVolume
(948,),    contrastLeft
(948,),    goCue_times
(948,),    feedbackType
(948,),    stimOn_times
(946,),    itiDuration
(948,),    probabilityLeft
(948,),    feedback_times
(948,),    stimOff_times
(948,),    goCueTrigger_times
(948,),    firstMovement_times
(948,),    response_times
(948,),    contrastRight
(948, 2),    intervals_bpod
(948, 2),    intervals
(948,),    choice


9 334 2d5f6d81-38c4-4bdc-ac3c-302ea4d5f46e
10 334 d839491f-55d8-4cbe-a298-7839208ba12b
11 334 4b7fbad4-f6de-43b4-9b15-c7c7ef44db4b
12 334 d2918f52-8280-43c0-924b-029b2317e62c
13 334 c99d53e6-c317-4c53-99ba-070b26673ac4
14 334 ecb5520d-1358-434c-95ec-93687ecd1396
15 334 85dc2ebd-8aaf-46b0-9284-a197aee8b16f
16 334 3663d82b-f197-4e8b-b299-7b803a155b84
17 334 5386aba9-9b97-4557-abcd-abc2da66b863


Inconsistent dimensions for object: None 
(1011,),    goCue_times
(1011, 2),    intervals
(1011,),    probabilityLeft
(1010,),    itiDuration
(1011, 2),    intervals_bpod
(1011,),    goCueTrigger_times
(1011,),    response_times
(1011,),    feedbackType
(1011,),    contrastRight
(1011,),    stimOn_times
(1011,),    feedback_times
(1011,),    firstMovement_times
(1011,),    stimOff_times
(1011,),    contrastLeft
(1011,),    choice
(1011,),    rewardVolume


18 334 83e77b4b-dfa0-4af9-968b-7ea0c7a0c7e4
19 334 4b00df29-3769-43be-bb40-128b1cba6d35
20 334 eef82e27-c20e-48da-b4b7-c443031649e3
21 334 0deb75fb-9088-42d9-b744-012fb8fc4afb


...

205 334 a9272cce-6914-4b45-a05f-9e925b4c472a


Inconsistent dimensions for object: None 
(735,),    contrastRight
(735,),    goCueTrigger_times
(735,),    response_times
(735, 2),    intervals_bpod
(735,),    stimOn_times
(735,),    contrastLeft
(735,),    rewardVolume
(735,),    feedbackType
(735,),    goCue_times
(735,),    firstMovement_times
(735,),    choice
(735,),    probabilityLeft
(735,),    feedback_times
(731,),    itiDuration
(735, 2),    intervals
(735,),    stimOff_times


206 334 03063955-2523-47bd-ae57-f7489dd40f15
207 334 1e45d992-c356-40e1-9be1-a506d944896f
208 334 4d8c7767-981c-4347-8e5e-5d5fffe38534
209 334 66d98e6e-bcd9-4e78-8fbb-636f7e808b29
210 334 f9860a11-24d3-452e-ab95-39e199f20a93
211 334 4ecb5d24-f5cc-402c-be28-9d0f7cb14b3a
212 334 6fb1e12c-883b-46d1-a745-473cde3232c8
213 334 6f09ba7e-e3ce-44b0-932b-c003fb44fb89
214 334 695a6073-eae0-49e0-bb0f-e9e57a9275b9
215 334 c6db3304-c906-400c-aa0f-45dd3945b2ea
216 334 f3ce3197-d534-4618-bf81-b687555d1883
217 334 88d24c31-52e4-49cc-9f32-6adbeb9eba87
218 334 22e04698-b974-4805-b241-3b547dbf37bf
219 334 7416f387-b302-4ca3-8daf-03b585a1b7ec
220 334 1928bf72-2002-46a6-8930-728420402e01
221 334 7cb81727-2097-4b52-b480-c89867b5b34c
222 334 f84045b0-ce09-4ace-9d11-5ea491620707
223 334 fc14c0d6-51cf-48ba-b326-56ed5a9420c3
224 334 45ef6691-7b80-4a43-bd1a-85fc00851ae8
225 334 75b6b132-d998-4fba-8482-961418ac957d
226 334 032452e9-1886-449d-9c13-0f192572e19f
227 334 239cdbb1-68e2-4eb0-91d8-ae5ae4001c7a
228 334 5c

Inconsistent dimensions for object: None 
(628,),    goCueTrigger_times
(628,),    stimOn_times
(628,),    probabilityLeft
(628,),    rewardVolume
(610,),    firstMovement_times
(628,),    feedback_times
(628,),    goCue_times
(628,),    choice
(628,),    response_times
(628,),    itiDuration
(628,),    feedbackType
(628,),    contrastRight
(628,),    contrastLeft
(628, 2),    intervals


300 334 d832d9f7-c96a-4f63-8921-516ba4a7b61f
301 334 dcceebe5-4589-44df-a1c1-9fa33e779727
302 334 b10ed1ba-2099-42c4-bee3-d053eb594f09
303 334 4ef13091-1bc8-4f32-9619-107bdf48540c
304 334 25f77e81-c1af-46ab-8686-73ac3d67c4a7
305 334 cae5cd75-55e5-4277-8db3-cf4d6c5ff918
306 334 f8041c1e-5ef4-4ae6-afec-ed82d7a74dc1
307 334 65f5c9b4-4440-48b9-b914-c593a5184a18
308 334 4ddb8a95-788b-48d0-8a0a-66c7c796da96
309 334 e12fbe11-8a6b-4bf6-a955-e5f6cec31ef1
310 334 a7763417-e0d6-4f2a-aa55-e382fd9b5fb8
311 334 037d75ca-c90a-43f2-aca6-e86611916779
312 334 5285c561-80da-4563-8694-739da92e5dd0
313 334 ff96bfe1-d925-4553-94b5-bf8297adf259
314 334 a9138924-4395-4981-83d1-530f6ff7c8fc
315 334 09394481-8dd2-4d5c-9327-f2753ede92d7
316 334 dc21e80d-97d7-44ca-a729-a8e3f9b14305
317 334 8c2f7f4d-7346-42a4-a715-4d37a5208535
318 334 952870e5-f2a7-4518-9e6d-71585460f6fe
319 334 f304211a-81b1-446f-a435-25e589fe3a5a
320 334 91e04f86-89df-4dec-a8f8-fa915c9a5f1a
321 334 73918ae1-e4fd-4c18-b132-00cb555b1ad2
322 334 c7

In [6]:
eid = '15f742e1-1043-45c9-9504-f1e8a53c1744'
one = ONE()  # mode='local'
atlas = AllenAtlas()

estimator = ESTIMATOR #(**ESTIMATOR_KWARGS)

subject = sessdf.xs(eid, level='eid').index[0]
subjeids = sessdf.xs(subject, level='subject').index.unique()
brainreg = dut.BrainRegions()
behavior_data = mut.load_session(eid, one=one)
pLeft_vec = np.array(behavior_data['probabilityLeft'])
try:
    tvec = dut.compute_target(TARGET, subject, subjeids, eid, MODELFIT_PATH,
                              modeltype=MODEL, beh_data=behavior_data,
                              one=one)
except ValueError:
    print('Model not fit.')
    tvec = dut.compute_target(TARGET, subject, subjeids, eid, MODELFIT_PATH,
                              modeltype=MODEL, one=one)

fvecs = [dut.compute_target(feature, subject, subjeids, eid, MODELFIT_PATH,
                          modeltype=MODEL, beh_data=behavior_data,
                          one=one) for feature in CONTROL_FEATURES]

try:
    trialsdf = bbone.load_trials_df(eid, one=one, addtl_types=['firstMovement_times'])
    if len(trialsdf) != len(tvec):
        raise IndexError
except IndexError:
    raise IndexError('Problem in the dimensions of dataframe of session')
trialsdf['react_times'] = trialsdf['firstMovement_times'] - trialsdf['goCue_times']

local md5 or size mismatch on dataset: cortexlab/Subjects/KS022/2019-12-09/001alf/_ibl_trials.response_times.npy


Downloading: /home/groups/sganguli/bensonb/Downloads/ONE/cache_dir/cortexlab/Subjects/KS022/2019-12-09/001/alf/_ibl_trials.response_times.e9ab5a29-9db4-4e8b-ac1f-ec82fdff65c0.npy Bytes: 6736


100%|█████████████████████████████████████████████████████████████████████████| 6736/6736 [00:00<00:00, 1137896.48it/s]
local md5 or size mismatch on dataset: cortexlab/Subjects/KS022/2019-12-09/001alf/_ibl_trials.response_times.npy


Downloading: /home/groups/sganguli/bensonb/Downloads/ONE/cache_dir/cortexlab/Subjects/KS022/2019-12-09/001/alf/_ibl_trials.response_times.e9ab5a29-9db4-4e8b-ac1f-ec82fdff65c0.npy Bytes: 6736


100%|█████████████████████████████████████████████████████████████████████████| 6736/6736 [00:00<00:00, 2523024.80it/s]


In [7]:
trialsdf.keys()

Index(['choice', 'probabilityLeft', 'feedbackType', 'feedback_times',
       'contrastLeft', 'contrastRight', 'goCue_times', 'stimOn_times',
       'firstMovement_times', 'trial_start', 'trial_end', 'react_times'],
      dtype='object')

In [12]:
def generate_shifts():
    out = np.arange(-10,10)
    np.random.shuffle(out)
    i = 0
    while i < len(out):
        yield out[i]
        i+=1
        
gs = generate_shifts()
for j in range(30):
    print(next(generate_shifts()))
    

-6
-1
-8
1
-1
7
4
5
7
-10
-5
-3
-5
-9
-1
6
-3
9
0
5
9
8
-7
-1
-3
9
-4
-10
9
1


In [3]:
# submit python file with eid inputs
for eid in sessdf_eids:
    slurm_utils.submit_python_file(py_file, 
                             eid, 
                             ADD_TO_SAVING_PATH = ADD_TO_SAVING_PATH,
                             n_gigs_ram = 8,
                             SLURM_DIRECTORY = SLURM_DIRECTORY,
                             SUBDIRECTORY = SUBDIRECTORY)
    

Submitted batch job 48163721
Submitted batch job 48163722
Submitted batch job 48163723
Submitted batch job 48163724
Submitted batch job 48163725
Submitted batch job 48163727
Submitted batch job 48163736
Submitted batch job 48163737
Submitted batch job 48163738
Submitted batch job 48163739
Submitted batch job 48163741
Submitted batch job 48163742
Submitted batch job 48163743
Submitted batch job 48163744
Submitted batch job 48163785
Submitted batch job 48163786
Submitted batch job 48163787
Submitted batch job 48163788
Submitted batch job 48163789
Submitted batch job 48163790
Submitted batch job 48163791
Submitted batch job 48163792
Submitted batch job 48163793
Submitted batch job 48163794
Submitted batch job 48163795
Submitted batch job 48163796
Submitted batch job 48163797
Submitted batch job 48163798
Submitted batch job 48163799
Submitted batch job 48163800
Submitted batch job 48163803
Submitted batch job 48163805
Submitted batch job 48163806
Submitted batch job 48163807
Submitted batc

Submitted batch job 48164113
Submitted batch job 48164115
Submitted batch job 48164116
Submitted batch job 48164118
Submitted batch job 48164119
Submitted batch job 48164120
Submitted batch job 48164121
Submitted batch job 48164122
Submitted batch job 48164123
Submitted batch job 48164124
Submitted batch job 48164125
Submitted batch job 48164126
Submitted batch job 48164127
Submitted batch job 48164128
Submitted batch job 48164129
Submitted batch job 48164130
Submitted batch job 48164131
Submitted batch job 48164132
Submitted batch job 48164133
Submitted batch job 48164134
Submitted batch job 48164136
Submitted batch job 48164137
Submitted batch job 48164138
Submitted batch job 48164139
Submitted batch job 48164140
Submitted batch job 48164141
Submitted batch job 48164142
Submitted batch job 48164143
Submitted batch job 48164144
Submitted batch job 48164145
Submitted batch job 48164146
Submitted batch job 48164147
Submitted batch job 48164149
Submitted batch job 48164150
Submitted batc

In [5]:

os.system("squeue -u $USER");
slurm_utils.get_decoding_output_files(SLURM_DIRECTORY = SLURM_DIRECTORY,
                                      SUBDIRECTORY = SUBDIRECTORY)

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          48164173    normal /home/gr  bensonb PD       0:00      1 (Priority)
          48164172    normal /home/gr  bensonb PD       0:00      1 (Priority)
          48164170    normal /home/gr  bensonb PD       0:00      1 (Priority)
          48164169    normal /home/gr  bensonb PD       0:00      1 (Priority)
          48164168    normal /home/gr  bensonb PD       0:00      1 (Priority)
          48164167    normal /home/gr  bensonb PD       0:00      1 (Priority)
          48164166    normal /home/gr  bensonb PD       0:00      1 (Priority)
          48164165    normal /home/gr  bensonb PD       0:00      1 (Priority)
          48164164    normal /home/gr  bensonb PD       0:00      1 (Priority)
          48164163    normal /home/gr  bensonb PD       0:00      1 (Priority)
          48164162    normal /home/gr  bensonb PD       0:00      1 (Priority)
          48164161    normal /home/gr  bensonb

[]

In [4]:
SUBDIRECTORY = 'decode_pLeft_task_Logistic_accuracy_control_10_impostor-session_align_goCue_times_timeWin_-0_4_-0_1_testnulltype1'

slurm_utils.gather_save_outputs(SUBDIRECTORY, SLURM_DIRECTORY, OUTPUT_PATH)